# Pediatric CXR — Wasserstein / Optimal Transport Domain Adaptation

**Objective:** leverage large labeled adult CXRs (NIH ChestX-ray14) to improve pneumonia classification on a small labeled pediatric dataset (Kermany et al.) using Optimal Transport for domain alignment.

**Architecture:**
- Shared `ImageEncoder` (ResNet18 or ViT-B/16) + frozen `TextEncoder` (BERT) — reused from `cxr_model.py`
- `SlicedWassersteinLoss` (or Sinkhorn) aligns adult/pediatric embeddings in the shared CLIP space
- Optional gradient-reversal adversarial discriminator for complementary alignment

**Training objective:**
$$\mathcal{L} = \mathcal{L}_{\text{CLIP-adult}} + \lambda_{\text{ped}}\mathcal{L}_{\text{CLIP-ped}} + \lambda_{\text{OT}}\cdot\text{SWD}(\mathbf{z}_{\text{adult}}, \mathbf{z}_{\text{ped}}) + \lambda_{\text{adv}}\mathcal{L}_{\text{domain}}$$


##Imports/Config

In [ ]:
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from cxr_data import load_path_remap
from cxr_engine import (
    build_transform,
    collect_image_embeddings,
    evaluate_clip_classifier,
    load_prompt_tensors,
    plot_tsne_before_after,
    prepare_loaders_from_manifests,
    stratified_subset_indices,
)
from cxr_eval_viz import run_evaluation_figures, plot_roc_comparison, plot_metrics_comparison_bars

from ot_model import build_ot_model, WassersteinDomainAdaptationModel
from ot_engine import (
    train_ot_domain_adapt,
    finetune_ot_pediatric,
    evaluate_ot_classifier,
    ot_learning_curve_experiment,
    save_ot_bundle,
)

print("PyTorch:", torch.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

In [ ]:
# Paths 
PROJECT     = Path('.')              
ADULT_CSV   = PROJECT / 'data/processed/adult_manifest.csv'
PED_CSV     = PROJECT / 'data/processed/pediatric_manifest.csv'
PROMPT_PT   = PROJECT / 'data/processed/bert_prompt_tokens.pt'
OUT_DIR     = PROJECT / 'outputs/ot'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparams
IMAGE_BACKBONE  = 'resnet18'  
IMAGE_SIZE      = 224
BATCH_SIZE      = 32
ADULT_EPOCHS    = 5
PED_EPOCHS      = 5
LR_ADULT        = 1e-4
LR_PED          = 1e-4
SEED            = 42

# OT hyperparameters
LAMBDA_OT       = 0.5     
LAMBDA_ADV      = 0.1     
LAMBDA_PED      = 1.0    
N_PROJECTIONS   = 256    
MAX_ADULT_SAMPLES = None  

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config OK')

## Data Loaders

In [ ]:
remap = load_path_remap()  

adult_loader, ped_train_loader, ped_val_loader, ped_test_loader = prepare_loaders_from_manifests(
    ADULT_CSV, PED_CSV,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    remap=remap,
    max_adult_samples=MAX_ADULT_SAMPLES,
)

print(f'Adult train batches : {len(adult_loader):,}')
print(f'Ped   train batches : {len(ped_train_loader):,}')
print(f'Ped   val   batches : {len(ped_val_loader):,}')
print(f'Ped   test  batches : {len(ped_test_loader):,}')

input_ids, attention_mask = load_prompt_tensors(PROMPT_PT, DEVICE)

## Build OT Model

In [ ]:
model = build_ot_model(
    image_backbone=IMAGE_BACKBONE,
    lambda_ot=LAMBDA_OT,
    lambda_adv=LAMBDA_ADV,
    lambda_ped=LAMBDA_PED,
    n_projections=N_PROJECTIONS,
    pretrained_image=True,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params/1e6:.1f}M  |  Trainable: {trainable/1e6:.1f}M')
print(f'λ_ot={LAMBDA_OT}  λ_adv={LAMBDA_ADV}  λ_ped={LAMBDA_PED}  n_proj={N_PROJECTIONS}')

## Adult Pre-Training w/ OT Alignment

Joint training: adult CLIP loss + pediatric CLIP loss + Sliced Wasserstein alignment + adversarial domain loss.

In [ ]:
history = train_ot_domain_adapt(
    model,
    adult_loader,
    ped_train_loader,
    input_ids,
    attention_mask,
    DEVICE,
    epochs=ADULT_EPOCHS,
    lr=LR_ADULT,
    reversal_schedule='linear',
    freeze_text=True,
)

In [ ]:
df_hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, title in zip(
    axes,
    ['clip_loss', 'ot_loss', 'adv_loss'],
    ['CLIP Loss (adult + ped)', 'OT Alignment Loss (SWD)', 'Adversarial Domain Loss'],
):
    ax.plot(df_hist['epoch'], df_hist[col], 'o-')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title(title)
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / 'pretrain_loss_curves.png', dpi=150)
plt.show()
print('Loss curves saved')

## t-SNE: Embedding Alignment Before / After Pre-Training

In [ ]:
from cxr_model import ImageTextModel
import copy


model_pretrained_init = build_ot_model(
    image_backbone=IMAGE_BACKBONE, pretrained_image=True
).to(DEVICE)

@torch.no_grad()
def collect_ped_embeddings(m):
    m.eval()
    zs, ys = [], []
    for x, y in ped_test_loader:
        zs.append(m.encode_image(x.to(DEVICE)).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(zs), np.concatenate(ys)

emb_before, labels_emb = collect_ped_embeddings(model_pretrained_init)
emb_after, _           = collect_ped_embeddings(model)

plot_tsne_before_after(
    emb_before, emb_after, labels_emb,
    out_path=OUT_DIR / 'tsne_before_after_ot_pretrain.png',
)
print('t-SNE saved')
del model_pretrained_init

In [ ]:
frozen_text_emb = finetune_ot_pediatric(
    model,
    ped_train_loader,
    input_ids,
    attention_mask,
    DEVICE,
    epochs=PED_EPOCHS,
    lr=LR_PED,
)


save_ot_bundle(
    OUT_DIR / 'ot_model_bundle.pt',
    model,
    frozen_text_emb,
    image_backbone=IMAGE_BACKBONE,
    image_size=IMAGE_SIZE,
    n_projections=N_PROJECTIONS,
    lambda_ot=LAMBDA_OT,
    lambda_adv=LAMBDA_ADV,
    lambda_ped=LAMBDA_PED,
    batch_size=BATCH_SIZE,
)
print('Bundle saved')

## Evaluation on Pediatric Test Set

In [ ]:
ev_ot = evaluate_ot_classifier(
    model, ped_test_loader, input_ids, attention_mask, DEVICE,
    frozen_text_emb=frozen_text_emb,
)
print(f'OT Model — AUC: {ev_ot.auc:.4f}   F1: {ev_ot.f1:.4f}')


metrics_ot = run_evaluation_figures(
    ev_ot.labels,
    ev_ot.probs_positive,
    OUT_DIR / 'eval_ot',
    prefix='ot_model',
)
print('\nFull metrics:')
for k, v in metrics_ot.items():
    print(f'  {k}: {v:.4f}')

##  Comparison -- OT vs CLIP-only vs Baseline


In [ ]:
from cxr_engine import (
    ImageTextModel, train_adult_clip, finetune_pediatric_clip,
    evaluate_clip_classifier, train_baseline, evaluate_baseline,
)

# CLIP only
model_clip = ImageTextModel(image_backbone=IMAGE_BACKBONE, pretrained_image=True).to(DEVICE)
train_adult_clip(
    model_clip, adult_loader, input_ids, attention_mask,
    DEVICE, epochs=ADULT_EPOCHS, lr=LR_ADULT,
)
text_emb_clip = finetune_pediatric_clip(
    model_clip, ped_train_loader, input_ids, attention_mask,
    DEVICE, epochs=PED_EPOCHS, lr=LR_PED,
)
ev_clip = evaluate_clip_classifier(
    model_clip, ped_test_loader, input_ids, attention_mask, DEVICE,
    frozen_text_emb=text_emb_clip,
)
print(f'CLIP-only — AUC: {ev_clip.auc:.4f}   F1: {ev_clip.f1:.4f}')

# baseline
torch.manual_seed(SEED)
model_bl = train_baseline(
    ped_train_loader, ped_val_loader, DEVICE,
    epochs=PED_EPOCHS, lr=LR_PED, backbone='resnet18',
)
ev_bl = evaluate_baseline(model_bl, ped_test_loader, DEVICE)
print(f'Baseline  — AUC: {ev_bl.auc:.4f}   F1: {ev_bl.f1:.4f}')

In [ ]:
# ROC 
plot_roc_comparison(
    ev_ot.labels,
    ev_ot.probs_positive,
    ev_clip.probs_positive,
    label_a='OT (Wasserstein)',
    label_b='CLIP-only',
    out_path=OUT_DIR / 'roc_ot_vs_clip.png',
    title='ROC: OT vs CLIP-only (pediatric test)',
)

plot_metrics_comparison_bars(
    models=['OT (Wasserstein)', 'CLIP-only', 'Baseline'],
    aucs=[ev_ot.auc, ev_clip.auc, ev_bl.auc],
    f1s=[ev_ot.f1, ev_clip.f1, ev_bl.f1],
    out_path=OUT_DIR / 'metrics_comparison.png',
)

print('Comparison figures saved')

summary = pd.DataFrame([
    {'Model': 'OT (Wasserstein)',  'AUC': ev_ot.auc,   'F1': ev_ot.f1},
    {'Model': 'CLIP-only',         'AUC': ev_clip.auc, 'F1': ev_clip.f1},
    {'Model': 'Baseline (ResNet)', 'AUC': ev_bl.auc,   'F1': ev_bl.f1},
])
print(summary.to_string(index=False))
summary.to_csv(OUT_DIR / 'model_comparison.csv', index=False)

## Learning Curve

In [ ]:
FRACTIONS = [0.1, 0.25, 0.5, 1.0]

curve = ot_learning_curve_experiment(
    adult_loader,
    ped_train_loader.dataset,
    ped_test_loader,
    PROMPT_PT,
    DEVICE,
    fractions=FRACTIONS,
    adult_epochs=ADULT_EPOCHS,
    ped_epochs=PED_EPOCHS,
    lr_adult=LR_ADULT,
    lr_ped=LR_PED,
    image_backbone=IMAGE_BACKBONE,
    baseline_epochs=PED_EPOCHS,
    lr_base=LR_PED,
    batch_size=BATCH_SIZE,
    seed=SEED,
    lambda_ot=LAMBDA_OT,
    lambda_adv=LAMBDA_ADV,
    lambda_ped=LAMBDA_PED,
)

In [ ]:
df_ot = pd.DataFrame(curve['proposed'])
df_bl = pd.DataFrame(curve['baseline'])

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
for df, name, style in [(df_ot, 'OT (Wasserstein)', 'o-'), (df_bl, 'Baseline', 's--')]:
    ax0.plot(df['fraction'], df['auc'], style, label=name)
    ax1.plot(df['fraction'], df['f1'],  style, label=name)
for ax, ylabel, title in [
    (ax0, 'AUC-ROC', 'Pediatric test — AUC-ROC'),
    (ax1, 'F1',      'Pediatric test — F1'),
]:
    ax.set_xlabel('Pediatric train fraction')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / 'learning_curve_ot.png', dpi=150)
plt.show()

df_ot.to_csv(OUT_DIR / 'lc_ot.csv', index=False)
df_bl.to_csv(OUT_DIR / 'lc_baseline.csv', index=False)
print("\nOT learning curve:")
print(df_ot.to_string(index=False))
print("\nBaseline learning curve:")
print(df_bl.to_string(index=False))

## t-SNE

In [ ]:

from ot_model import load_ot_bundle

model_ft, text_emb_ft, meta = load_ot_bundle(OUT_DIR / 'ot_model_bundle.pt', DEVICE)

emb_pre_ft, labels_ft = collect_ped_embeddings(model) 
emb_post_ft, _        = collect_ped_embeddings(model_ft)

plot_tsne_before_after(
    emb_pre_ft, emb_post_ft, labels_ft,
    out_path=OUT_DIR / 'tsne_before_after_ot_finetune.png',
)
print('t-SNE (pre/post fine-tune) saved')

## Ablation (Effect of λ_OT)

In [ ]:
lambda_ot_values = [0.0, 0.1, 0.5, 1.0, 2.0]
ablation_results = []

for lam in lambda_ot_values:
    torch.manual_seed(SEED)
    m = build_ot_model(
        image_backbone=IMAGE_BACKBONE,
        lambda_ot=lam,
        lambda_adv=0.0,  
        lambda_ped=LAMBDA_PED,
        pretrained_image=True,
    ).to(DEVICE)


    train_ot_domain_adapt(
        m, adult_loader, ped_train_loader,
        input_ids, attention_mask, DEVICE,
        epochs=2, lr=LR_ADULT, verbose=False,
    )
    ft = finetune_ot_pediatric(
        m, ped_train_loader, input_ids, attention_mask, DEVICE,
        epochs=PED_EPOCHS, lr=LR_PED, verbose=False,
    )
    ev = evaluate_ot_classifier(m, ped_test_loader, input_ids, attention_mask, DEVICE, frozen_text_emb=ft)
    ablation_results.append({'lambda_ot': lam, 'auc': ev.auc, 'f1': ev.f1})
    print(f'  λ_ot={lam:.1f}  AUC={ev.auc:.4f}  F1={ev.f1:.4f}')
    del m

df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv(OUT_DIR / 'ablation_lambda_ot.csv', index=False)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 4))
ax0.plot(df_ablation['lambda_ot'], df_ablation['auc'], 'o-', color='steelblue')
ax0.set_xlabel('λ_OT'); ax0.set_ylabel('AUC-ROC'); ax0.set_title('Ablation: λ_OT vs AUC')
ax0.grid(True, alpha=0.3)
ax1.plot(df_ablation['lambda_ot'], df_ablation['f1'], 's-', color='coral')
ax1.set_xlabel('λ_OT'); ax1.set_ylabel('F1'); ax1.set_title('Ablation: λ_OT vs F1')
ax1.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / 'ablation_lambda_ot.png', dpi=150)
plt.show()
print('Ablation done')

## How Wasserstein Distance Changes

Plot how the SWD between adult and pediatric embeddings changes across training epochs to verify alignment is happening.

In [ ]:
from ot_model import SlicedWassersteinLoss

swd_fn = SlicedWassersteinLoss(n_projections=256).to(DEVICE)

@torch.no_grad()
def measure_swd(m, n_batches=20):
    m.eval()
    adult_z, ped_z = [], []
    a_iter = iter(adult_loader)
    p_iter = iter(ped_test_loader)
    for _ in range(min(n_batches, len(adult_loader), len(ped_test_loader))):
        try:
            xa, _ = next(a_iter)
            xp, _ = next(p_iter)
        except StopIteration:
            break
        adult_z.append(m.encode_image(xa.to(DEVICE)))
        ped_z.append(m.encode_image(xp.to(DEVICE)))
    za = torch.cat(adult_z)
    zp = torch.cat(ped_z)
    return swd_fn(za, zp).item()

m_init = build_ot_model(image_backbone=IMAGE_BACKBONE, n_projections=N_PROJECTIONS, pretrained_image=True).to(DEVICE)
swd_init = measure_swd(m_init)
swd_ot   = measure_swd(model_ft)
del m_init

print(f'SWD (ImageNet init)  : {swd_init:.6f}')
print(f'SWD (OT fine-tuned)  : {swd_ot:.6f}')
print(f'Relative reduction   : {100*(swd_init - swd_ot)/max(swd_init, 1e-8):.1f}%')

## Summary

In [ ]:
print('='*60)
print('FINAL RESULTS — Pediatric CXR Test Set')
print('='*60)
print(summary.to_string(index=False))
print()
print(f'SWD reduction: {swd_init:.4f} → {swd_ot:.4f}  '
      f'({100*(swd_init-swd_ot)/max(swd_init,1e-8):.1f}% ↓)')
print()
print('All outputs written to:', OUT_DIR.resolve())